In [2]:
import os
import urllib.request
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential
import xgboost as xgb

# Set directories
SAVED_DIR = os.path.join("src", "models", "saved")
os.makedirs(SAVED_DIR, exist_ok=True)

# 1. Load Data
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv"
df = pd.read_csv(url, header=0, index_col=0)
df = pd.get_dummies(df, columns=["cbwd"])
df = df.dropna()

# ----------------------------------------------------------------------
# 2. TRAIN XGBOOST BASELINE (Matches R² = 0.6391, RMSE = 56.39)
# ----------------------------------------------------------------------
y_xgb = df["pm2.5"]
X_xgb = df.drop("pm2.5", axis=1)

X_train_x, X_test_x, y_train_x, y_test_x = train_test_split(
    X_xgb, y_xgb, test_size=0.2, random_state=42
)

model_xgb = xgb.XGBRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42
)
model_xgb.fit(X_train_x, y_train_x)

pred_xgb = model_xgb.predict(X_test_x)
rmse_xgb = np.sqrt(mean_squared_error(y_test_x, pred_xgb))
r2_xgb = r2_score(y_test_x, pred_xgb)

print("=" * 45)
print(f"XGBoost Result: RMSE = {rmse_xgb:.2f} µg/m³ | R² = {r2_xgb:.4f}")
print("=" * 45)

model_xgb.save_model(os.path.join(SAVED_DIR, "xgb_model.json"))
joblib.dump(list(X_xgb.columns), os.path.join(SAVED_DIR, "xgb_features.pkl"))

# ----------------------------------------------------------------------
# 3. TRAIN LSTM MODEL (Matches R² = 0.9523, RMSE = 20.61)
# ----------------------------------------------------------------------
# Ensure target PM2.5 is at index 0
cols = ["pm2.5"] + [c for c in df.columns if c != "pm2.5"]
df_reordered = df[cols]

scaler = StandardScaler()
data_scaled = scaler.fit_transform(df_reordered.values)


# Look-back window of 24h including all lagged channels
def create_dataset(dataset, look_back=24):
  X, y = [], []
  for i in range(len(dataset) - look_back):
    X.append(dataset[i : (i + look_back), :])
    y.append(dataset[i + look_back, 0])
  return np.array(X), np.array(y)


LOOK_BACK = 24
X_series, y_series = create_dataset(data_scaled, LOOK_BACK)

# Chronological split for sequence evaluation
train_size = int(len(X_series) * 0.8)
X_train_l, X_test_l = X_series[:train_size], X_series[train_size:]
y_train_l, y_test_l = y_series[:train_size], y_series[train_size:]

model_lstm = Sequential([
    Input(shape=(X_train_l.shape[1], X_train_l.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])

model_lstm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse"
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
model_lstm.fit(
    X_train_l,
    y_train_l,
    validation_split=0.1,
    epochs=30,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1,
)

# Unscale predictions using index 0 parameters
pred_scaled_l = model_lstm.predict(X_test_l).flatten()
y_test_unscaled = y_test_l * np.sqrt(scaler.var_[0]) + scaler.mean_[0]
pred_unscaled_l = pred_scaled_l * np.sqrt(scaler.var_[0]) + scaler.mean_[0]

rmse_lstm = np.sqrt(mean_squared_error(y_test_unscaled, pred_unscaled_l))
r2_lstm = r2_score(y_test_unscaled, pred_unscaled_l)

print("=" * 45)
print(f"LSTM Result:    RMSE = {rmse_lstm:.2f} µg/m³ | R² = {r2_lstm:.4f}")
print("=" * 45)

# Save artifacts
model_lstm.save(os.path.join(SAVED_DIR, "lstm_model.h5"))
joblib.dump(scaler, os.path.join(SAVED_DIR, "feature_scaler.pkl"))
# Also save scaler as target scaler for backward compatibility
joblib.dump(scaler, os.path.join(SAVED_DIR, "target_scaler.pkl"))

print("\nArtifact verification:")
for f in [
    "xgb_model.json",
    "xgb_features.pkl",
    "lstm_model.h5",
    "feature_scaler.pkl",
    "target_scaler.pkl",
]:
  print(f" - {f}: {os.path.exists(os.path.join(SAVED_DIR, f))}")

XGBoost Result: RMSE = 56.43 µg/m³ | R² = 0.6386
Epoch 1/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 34s 82ms/step - loss: 0.2567 - val_loss: 0.1019
Epoch 2/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - loss: 0.1176 - val_loss: 0.0667
Epoch 3/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 14s 59ms/step - loss: 0.0958 - val_loss: 0.0566
Epoch 4/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - loss: 0.0851 - val_loss: 0.0546
Epoch 5/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 12s 53ms/step - loss: 0.0816 - val_loss: 0.0522
Epoch 6/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 54ms/step - loss: 0.0794 - val_loss: 0.0494
Epoch 7/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 14s 58ms/step - loss: 0.0783 - val_loss: 0.0504
Epoch 8/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - loss: 0.0774 - val_loss: 0.0540
Epoch 9/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 56ms/step - loss: 0.0766 - val_loss: 0.0499
Epoch 10/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 54ms/step - loss: 0.0743 - val_loss: 0.0531
Epoch 11/30
235/235 ━━━━━━━━━━━━━━━━━━━━ 13s 55ms/step - loss: 0.0754 

LSTM Result:    RMSE = 20.60 µg/m³ | R² = 0.9523

Artifact verification:
 - xgb_model.json: True
 - xgb_features.pkl: True
 - lstm_model.h5: True
 - feature_scaler.pkl: True
 - target_scaler.pkl: True


In [5]:
model_lstm.save_weights(os.path.join(SAVED_DIR, "lstm_model.weights.h5"))

In [6]:
import os
import joblib

SAVED_DIR = os.path.join("src", "models", "saved")
os.makedirs(SAVED_DIR, exist_ok=True)

# 1. Save weights using layer-by-layer numpy dump (100% version-safe across any Keras/TF version)
weights_list = model_lstm.get_weights()
joblib.dump(weights_list, os.path.join(SAVED_DIR, "lstm_model_weights.pkl"))
print("Saved: lstm_model_weights.pkl (Python pickle format, zero layer-count mismatch)")

Saved: lstm_model_weights.pkl (Python pickle format, zero layer-count mismatch)
